# Harmony of the Spheres
Generates a daily playlist based on your astrological chart for the day

**Pipeline:**
1. Compute natal + transit chart from birth info
2. Filter active aspects by orb limits
3. Call Gemini to select the most significant aspects and generate a horoscope
4. Build a hand-coded audio feature target vector from selected aspects
5. Load music library
6. Refine the target vector using a personal Lasso model trained on liked songs
7. Score and rank tracks against the blended target vector

## 1 — Imports & Config

In [ ]:
%pip install kerykeion google-genai scikit-learn requests --quiet

In [ ]:
import sys
import os
import pandas as pd
from pathlib import Path

# add src package to path
sys.path.insert(0, '/Users/meghanapakala/Desktop/astral_audio/src')

from aspects import get_transit_aspects, format_aspects
from horoscope import get_horoscope, get_select_aspects
from library import load_music_library
from score import build_target_vector, score_tracks
from model import train_model, predict_target_vector, blend_target_vectors, save_model, load_model

In [ ]:
# --- File Paths ---
BASE_PATH          = '/Users/meghanapakala/Desktop/astral_audio/music_library'
LOCAL_LIBRARY_PATH = f'{BASE_PATH}/local_library.csv'
USER_PLAYLIST_PATH = f'{BASE_PATH}/Liked_Songs.csv'  # set to None to use local pool
MODEL_PATH         = '/Users/meghanapakala/Desktop/astral_audio/personal_model.pkl'

# --- Birth Info ---
# in web app these come from input form
# lat/lng from Google Places Autocomplete
birth_info = {
    'date': '1995-11-12',
    'time': '06:30',
    'lat':  14.4426,
    'lng':  79.9865,
    'tz':   'Asia/Kolkata'
    }

current_loc = {
    'lat': 34.0195,
    'lng': -118.4912,
    'tz':  'America/Los_Angeles'
    }

# --- Gemini API Keys ---
# set BYPASS_GEMINI=1 to skip the API call and use the stub horoscope
os.environ['GEMINI_API_KEY']   = ''   # primary key
os.environ['GEMINI_API_KEY_2'] = ''   # backup key (optional)
os.environ['BYPASS_GEMINI']    = '1'  # set to '0' to call Gemini

## 2 — Natal & Transit Aspects

Builds a natal chart from birth data and a transit chart from the current datetime and location. Filters to five aspect types (conjunction, opposition, trine, square, sextile) within moiety orb limits — each planet gets its own orb based on orbital speed, and the limit for any aspect is the average of the two planets' orbs.

In [ ]:
daily_aspects, natal_chart, transit_chart = get_transit_aspects(birth_info, transit_loc=current_loc)

print(f'Active aspects ({len(daily_aspects)}):')
print(format_aspects(daily_aspects))

## 3 — LLM Horoscope

Passes all filtered aspects to Gemini. The LLM selects the 3 most significant aspects for today's emotional and sonic character, prioritizing personal planets and coherent themes over raw orb tightness. Returns a horoscope narrative and daily keywords alongside the selected aspects.

In [ ]:
horoscope = get_horoscope(daily_aspects)

print('Daily Summary:')
print(horoscope['daily_summary'])
print()
print('Keywords:', ', '.join(horoscope['daily_keywords']))
print()
print('Aspects:')
for a in horoscope['aspects']:
    print(f"\n{a['aspect']}")
    print(f"{a['meaning']}")

## 4 — Hand-Coded Target Vector

Each planet has a manually defined audio profile (valence, energy, danceability, acousticness, tempo, mode) based on its astrological character. The LLM-selected aspects are averaged into a single target vector, weighted by orb tightness — tighter orbs carry more weight via inverse square law.

This vector represents the astrological "ideal" audio profile for today, independent of any individual user's taste. It serves as the primary signal and the baseline for the ML refinement step.

In [ ]:
select_aspects = get_select_aspects(daily_aspects, horoscope)
handcoded_vector = build_target_vector(select_aspects)

print('Select Aspects:')
for a in select_aspects:
    print(f'  Natal {a.p1_name} in {a.aspect} with transiting {a.p2_name} '
          f'(orb: {abs(a.orbit):.2f}°)')

print()
print('Hand-Coded Target Vector:')
for feature, value in handcoded_vector.items():
    if feature == 'tempo':
        print(f'  {feature:<16}: {value:.1f} BPM')
    elif feature == 'mode':
        label = {1: 'major', 0: 'minor', None: 'no preference'}.get(value)
        print(f'  {feature:<16}: {label}')
    else:
        print(f'  {feature:<16}: {value:.3f}')

## 5 — Music Library

Loads tracks from one of two sources:

**Exportify CSV (preferred)**  
Export a playlist (Liked Songs) from [exportify.net](https://exportify.net) and set `USER_PLAYLIST_PATH` in config. New tracks are merged into the local library pool.

**Local Library (fallback)**  
Set `USER_PLAYLIST_PATH = None` to use the local pool directly.

In [ ]:
library = load_music_library(
    user_playlist_path = USER_PLAYLIST_PATH,
    local_library_path = LOCAL_LIBRARY_PATH
    )

print(f'Library loaded: {len(library)} tracks')

## 6 — Personal Lasso Model

The hand-coded vector captures universal astrological logic but knows nothing about the individual user's taste. The Lasso model personalizes it.

**Training signal:** Each liked song has an `added_at` date — the day the user saved it to Spotify. This is treated as a weak proxy for mood: the assumption is that the audio features of a saved song reflect what the user wanted to hear that day.

**Feature engineering:** For each save date, the active aspects are encoded as a 500-dimensional sparse vector — one slot per possible (natal planet, aspect type, transit planet) combination, valued at `1 / (orb² + 1)` so tighter orbs carry more weight. Most slots are 0 on any given day.

**Target:** The deviation of each song's audio features from the user's personal mean. Training on deltas rather than raw values anchors the model to the user's taste range and makes the learning task easier — the model only needs to explain mood shifts, not absolute preferences.

**Model:** One `LassoCV` per audio feature. Lasso is well-suited here because the feature space is sparse (few aspects active at once), the dataset is small (~2000 songs), and the built-in L1 penalty zeroes out aspect combinations that aren't predictive — acting as automatic feature selection.

**Blend:** The model prediction is blended with the hand-coded vector at 30/70. This is intentionally conservative — the save date signal is noisy and R² is low across features (~0.001–0.065). The blend lets the model nudge the output toward the user's personal taste range without overriding the astrological logic that drives the playlist concept.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)

# train if liked songs export has added_at dates, otherwise load existing model
if os.environ.get('BYPASS_GEMINI') == '1':
    # stub mode — skip model, use hand-coded vector only
    model_bundle = None
    print('Stub mode — skipping model.')

elif USER_PLAYLIST_PATH and Path(USER_PLAYLIST_PATH).exists():
    try:
        raw_df = pd.read_csv(USER_PLAYLIST_PATH)
        raw_df.columns = [c.lower().replace(' ', '_').replace('(s)', 's') for c in raw_df.columns]
        if 'added_at' in raw_df.columns:
            model_bundle = train_model(raw_df, birth_info)
            save_model(model_bundle, MODEL_PATH)
            print('Model trained and saved.')
        else:
            model_bundle = load_model(MODEL_PATH) if Path(MODEL_PATH).exists() else None
            print('No added_at column — loaded existing model.' if model_bundle else 'No added_at column and no saved model.')
    except Exception as e:
        model_bundle = load_model(MODEL_PATH) if Path(MODEL_PATH).exists() else None
        print(f'Training failed ({e}) — loaded existing model.' if model_bundle else f'Training failed ({e}) — falling back to hand-coded vector.')

elif Path(MODEL_PATH).exists():
    model_bundle = load_model(MODEL_PATH)
    print('Loaded existing model.')

else:
    model_bundle = None
    print('No liked songs upload and no saved model — using hand-coded vector.')

if model_bundle:
    print()
    print('User mean:')
    for f, v in model_bundle['user_mean'].items():
        print(f'  {f:<16}: {v:.3f}')

## 7 — Score Tracks & Output Playlist

Tracks are scored by weighted Euclidean distance from the target vector — lower score means closer match. Valence and energy are weighted highest as the most perceptually salient features. Mode uses a soft penalty rather than a hard filter so it nudges ranking without eliminating songs.

The final target vector is either:
- **Blended** (30% model prediction + 70% hand-coded) if a trained model is available
- **Hand-coded only** if no model exists or the pipeline is in stub mode

In [ ]:
# blend model prediction with hand-coded vector if model is available
if model_bundle is not None and daily_aspects:
    model_vector = predict_target_vector(daily_aspects, model_bundle)
    target_vector = blend_target_vectors(model_vector, handcoded_vector, model_weight=0.3)
    print('Target vector: blended (30% model, 70% hand-coded)')
else:
    target_vector = handcoded_vector
    print('Target vector: hand-coded only')

print()
for feature, value in target_vector.items():
    if feature == 'tempo':
        print(f'  {feature:<16}: {value:.1f} BPM')
    elif feature == 'mode':
        label = {1: 'major', 0: 'minor', None: 'no preference'}.get(value)
        print(f'  {feature:<16}: {label}')
    else:
        print(f'  {feature:<16}: {value:.3f}')

print()
playlist = score_tracks(library, target_vector)

print('=' * 55)
print('YOUR DAILY ASTRO PLAYLIST')
print('=' * 55)
print()
print(horoscope['daily_summary'])
print()
print(f"Today's energy: {', '.join(horoscope['daily_keywords'])}")
print()
for i, row in playlist.iterrows():
    print(f"{i+1:>2}. {row['track_name']} — {row['artist_names']}  (score: {row['score']:.3f})")